# Route A — Object Detection with YOLO
## Core code samples, Sessions 1–9

**Teacher notes are marked with `TEACHER NOTE`. Delete those cells before handing this to students.**

Two bugs are planted on purpose. They are in Session 2 and Session 4, and both are
flagged in a teacher note directly above them.


---
# Session 1 — Meet the AI that sees

**Predict:** if you had to teach a robot to find the water bottle in this picture,
what would you tell it?


### Teacher code — run this, do not study it

In [ ]:
from ultralytics import YOLO
import cv2
from IPython.display import Image, display

model = YOLO("yolov26n.pt")
print("Model is ready.")

### Run — find things in a picture

In [ ]:
picture = cv2.imread("desk.jpg")
results = model(picture, verbose=False)

for box in results[0].boxes:
    class_id = int(box.cls[0])
    name = model.names[class_id]
    print(name)

### Investigate — how many things does it know?

In Term 4 our fruit classifier knew **2** things, and we typed in **4** rows of data by hand.

How many does this one know?

In [ ]:
print(len(model.names))
print(model.names)

> **TEACHER NOTE.** Land this point out loud: nobody typed in the pictures.
> Somebody showed this model around 200,000 photographs. That is the whole
> difference between Term 4 and this term.

### Make
Photograph three objects around the room. Run the code on each one.
Write down what worked and what did not.

In [ ]:
picture = cv2.imread("my_photo.jpg")
results = model(picture, verbose=False)

for box in results[0].boxes:
    print(model.names[int(box.cls[0])])

---
# Session 2 — How sure is it?

**Predict:** yesterday the model got some things wrong.
Did it say nothing, or did it say something *quietly*?


### Run — print the confidence too

In [ ]:
picture = cv2.imread("desk.jpg")
results = model(picture, verbose=False)

for box in results[0].boxes:
    name = model.names[int(box.cls[0])]
    conf = float(box.conf[0])
    print(name, conf)

### Investigate

In Term 4, three neighbours voted:

- all three said "apple" — we were **sure**
- two said apple, one said watermelon — we were **less sure**

Confidence is that same feeling, written as a decimal between 0 and 1.

> **TEACHER NOTE — PLANTED BUG (silent).**
> The threshold below is `0.9`. On most classroom objects, nothing prints.
> No error, no red text — just an empty cell.
>
> Let them sit with it. Hint at about four minutes: *"the code ran fine,
> so what did it decide?"* Do not point at the number.

### Modify — only show the ones it is sure about

In [ ]:
picture = cv2.imread("desk.jpg")
results = model(picture, verbose=False)

for box in results[0].boxes:
    name = model.names[int(box.cls[0])]
    conf = float(box.conf[0])

    if conf > 0.9:
        print(name, conf)

### Make
Find one object that scores above 0.8, and one that scores below 0.4.
Write one sentence about why the model was less sure.

---
# Session 3 — Where is it?

**Predict:** here is a box drawn on a picture.
Give me **four numbers** that describe where it is.


### Run — print the corners

In [ ]:
picture = cv2.imread("desk.jpg")
results = model(picture, verbose=False)

box = results[0].boxes[0]

x1 = int(box.xyxy[0][0])
y1 = int(box.xyxy[0][1])
x2 = int(box.xyxy[0][2])
y2 = int(box.xyxy[0][3])

print(x1, y1, x2, y2) 

### Investigate

In Term 2, `cv2.boundingRect` gave us `x, y, w, h` — **one corner and a size**.

YOLO gives us `x1, y1, x2, y2` — **two corners**.

Same box. Different description.

**Your turn.** Do not look this up. Work out how to get `w` from `x1` and `x2`.

### Modify — width and centre

In [ ]:
w = x2 - x1
h = y2 - y1
center_x = (x1 + x2) // 2

print("width:", w)
print("height:", h)
print("centre x:", center_x)

### Make — paper task, laptops closed
Five printed frames. Read off the coordinates by hand and work out
the width and the centre for each one.

---
# Session 4 — Many things at once

**Predict:** the table has one bottle on it. Now I add three more objects.
What changes about what the code gets back?


### Run — loop through everything it found

In [ ]:
picture = cv2.imread("messy_desk.jpg")
results = model(picture, verbose=False)

print("Things found:", len(results[0].boxes))

for box in results[0].boxes:
    name = model.names[int(box.cls[0])]
    conf = float(box.conf[0])
    print(name, conf)

### Investigate

Every vision function we wrote in Term 2 and Term 3 ended like this:

```python
return cv2.boundingRect(max(contours, key=cv2.contourArea))
```

Exactly **one** object. Always.

This one is different. Sometimes zero. Sometimes eleven.

> **TEACHER NOTE — PLANTED BUG (loud).**
> The cell below grabs `boxes[0]` without checking. Point the camera at a
> blank wall and it crashes with an IndexError.
>
> This is deliberately right after last session's silent bug. Ask them:
> *which was easier to find? Which one was more dangerous?*

In [ ]:
picture = cv2.imread("blank_wall.jpg")
results = model(picture, verbose=False)

box = results[0].boxes[0]
print(model.names[int(box.cls[0])])

### Modify — check before you look

In [ ]:
picture = cv2.imread("blank_wall.jpg")
results = model(picture, verbose=False)

if len(results[0].boxes) == 0:
    print("I cannot see anything.")
else:
    box = results[0].boxes[0]
    print(model.names[int(box.cls[0])])

### Make — object census
Point the camera at a messy desk. Count how many of each thing it finds.

`.count()` is back from Term 4, Session 1.

In [ ]:
picture = cv2.imread("messy_desk.jpg")
results = model(picture, verbose=False)

names = []
for box in results[0].boxes:
    names.append(model.names[int(box.cls[0])])

print(names)
print("bottles:", names.count("bottle"))
print("cups:", names.count("cup"))

---
# Session 5 — Choosing one

**Predict:** we can now find every bottle in the picture.
If there are three of them, which one should the robot drive to?


### Run — only the ones with the right name

In [ ]:
picture = cv2.imread("messy_desk.jpg")
results = model(picture, verbose=False)

for box in results[0].boxes:
    name = model.names[int(box.cls[0])]
    conf = float(box.conf[0])

    if name == "bottle":
        print(name, conf)

### Investigate

Two filters have to stack:

1. the right **name**
2. enough **confidence**

Neither one on its own is enough. A wrong guess at 0.3 confidence is still a
wrong guess, and a correct guess about the wrong object is still the wrong object.

### Make — the big one

This function is the deliverable of the whole first half of term.

Look at its name and what it gives back. Now look at `find_ball()` from Term 2.

In [ ]:
def find_target(frame, target_name):
    results = model(frame, imgsz=320, verbose=False)

    best_box = None
    best_conf = 0.5

    for box in results[0].boxes:
        name = model.names[int(box.cls[0])]
        conf = float(box.conf[0])

        if name == target_name and conf > best_conf:
            best_conf = conf
            best_box = box

    if best_box is None:
        return None

    x1 = int(best_box.xyxy[0][0])
    y1 = int(best_box.xyxy[0][1])
    x2 = int(best_box.xyxy[0][2])
    y2 = int(best_box.xyxy[0][3])

    return x1, y1, x2 - x1, y2 - y1

> **TEACHER NOTE.** `best_conf` starts at `0.5`, so it is doing two jobs at once:
> it is the confidence threshold **and** the running best score. Worth pointing at
> once students have it working — it is a genuinely neat trick and they wrote it.

### Check it works

In [ ]:
picture = cv2.imread("messy_desk.jpg")
result = find_target(picture, "bottle")

if result is None:
    print("No bottle here.")
else:
    x, y, w, h = result
    print("Found a bottle at", x, y, "size", w, h)

---
# Session 6 — Bringing it to life

Nothing new today. Every piece below is from Term 2.


### Make — live detection on the laptop camera

In [ ]:
import time
import cv2
from IPython.display import display, Image

camera = cv2.VideoCapture(0, cv2.CAP_AVFOUNDATION)

ok, frame = camera.read()
ok, frame_jpg = cv2.imencode(".jpg", frame)
handle = display(Image(frame_jpg.tobytes()), display_id=True)

try:
    while True:
        ok, frame = camera.read()
        if not ok:
            continue

        result = find_target(frame, "bottle")

        if result:
            x, y, w, h = result
            cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
            cv2.putText(frame, f"bottle w:{w}", (x, y - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

        ok, frame_jpg = cv2.imencode(".jpg", frame)
        if ok:
            handle.update(Image(frame_jpg.tobytes()))

        time.sleep(0.05)

finally:
    camera.release()
    print("Camera closed.")

### Modify
Change `imgsz=320` inside `find_target` to `imgsz=640`. Watch the speed.
Then change it back. What did we gain, and what did we lose?

---
# Session 7 — Box to number

The robot still does not move today.

**Predict:** a bottle at the far left of the picture, then the centre, then the far right.
If we made a number out of the box position, what should that number do?


In [ ]:
import time
import cv2
from IPython.display import display, Image

camera = cv2.VideoCapture(0, cv2.CAP_AVFOUNDATION)

try:
    while True:
        ok, frame = camera.read()
        if not ok:
            continue

        result = find_target(frame, "bottle")

        if result is None:
            print("Show me a bottle.")
            time.sleep(0.5)
            continue

        x, y, w, h = result
        center_x = x + w // 2

        error_x = center_x - 320
        error_y = 200 - w

        print(f"centre:{center_x}  error_x:{error_x}  error_y:{error_y}")
        time.sleep(0.2)

finally:
    camera.release()

### Investigate — the reveal

Term 3, colour tracking:

```python
error_x = center_x - frame_center
```

Today, YOLO:

```python
error_x = center_x - frame_center
```

They are the same line.

The controller never learned anything new. We only changed **where the box comes from**.

---
# Session 8 — The robot chases

Now we swap `find_target` into the tracker you already built in Term 3.
The controller below is **unchanged**. Do not tidy it up.


In [ ]:
def target_centralizing_approaching_Kp(target_name):
    import time
    import cv2
    import numpy as np
    from IPython.display import display, Image

    # Kp values
    Kp_x = 0.06
    Kp_y = 0.3

    frame_center = 320
    target_w = 200
    dead_zone_x = 30
    dead_zone_y = 10

    # function 1: get the bgr image
    def picture_bgr():
        picture_raw = got.read_camera_data()
        if picture_raw is None:
            return None
        return cv2.imdecode(np.frombuffer(picture_raw, np.uint8), cv2.IMREAD_COLOR)

    # function 2: THE SWAP - this used to be find_ball()
    def find_it(frame):
        return find_target(frame, target_name)

    # function 3: Kp move and bounding box
    def move_and_draw(frame, x, y, w, h):
        center_x = x + w // 2

        error_x = center_x - frame_center
        error_y = target_w - w

        if abs(error_x) < dead_zone_x and abs(error_y) < dead_zone_y:
            got.mecanum_stop()
            return True

        speed_x = int(Kp_x * error_x)
        speed_y = int(Kp_y * error_y)

        speed_x = max(min(speed_x, 15), -15)
        speed_y = max(min(speed_y, 15), -15)

        got.mecanum_move_xyz(speed_x, speed_y, 0)

        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
        cv2.putText(frame, f"{target_name} ex:{error_x} ey:{error_y}", (x, y - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    # main program
    got.open_camera()
    ok, picture_jpg = cv2.imencode(".jpg", picture_bgr())
    handle = display(Image(picture_jpg.tobytes()), display_id=True)

    try:
        while True:
            frame = picture_bgr()
            if frame is None:
                continue

            result = find_it(frame)
            if result:
                if move_and_draw(frame, *result):
                    break
            else:
                got.mecanum_stop()

            ok, picture_jpg = cv2.imencode(".jpg", frame)
            if ok:
                handle.update(Image(picture_jpg.tobytes()))

            time.sleep(0.05)

    finally:
        got.mecanum_stop()
        print("Robot stopped.")

In [ ]:
target_centralizing_approaching_Kp("bottle")

### Investigate
The robot overshoots more than it did with colour tracking. Why?

### Modify
Make `Kp_x` smaller. Make the dead zone wider. Tune it by feel.

---
# Session 9 — Capstone: the Lost Property Robot

The robot is given a list of lost items. It turns until it spots one,
drives up to it, and announces what it found.

Nothing new. Just assembly.


In [ ]:
import time
import numpy as np
import cv2

lost_items = ["bottle", "cup", "book"]

def picture_bgr():
    picture_raw = got.read_camera_data()
    if picture_raw is None:
        return None
    return cv2.imdecode(np.frombuffer(picture_raw, np.uint8), cv2.IMREAD_COLOR)

def search_for_lost_item():
    got.open_camera()
    got.play_audio_tts("I am looking for lost property.", 1, True)

    try:
        while True:
            got.mecanum_turn_speed(2, 15)
            frame = picture_bgr()
            if frame is None:
                continue

            for item in lost_items:
                if find_target(frame, item):
                    got.mecanum_stop()
                    print(f"I found a {item}!")
                    got.play_audio_tts(f"I found a {item}.", 1, True)
                    return item

            time.sleep(0.1)

    finally:
        got.mecanum_stop()

In [ ]:
found_item = search_for_lost_item()
time.sleep(0.5)
target_centralizing_approaching_Kp(found_item)
got.play_audio_tts(f"Here is your {found_item}. Job done!", 1, True)

### Extension — only if the simple version already runs
Deliver the item to an AprilTag zone using `check_point_navigation()` from Term 1.